<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/src/pnp_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HST 6-DoF pose evaluation

Estimate Hubble's camera-relative pose using the trained YOLOv8s-Pose keypoints and `src/pnp.py`. Then compare the estimates with the metadata and repeat PnP using ground-truth 2D keypoints as a geometry check.

This is the original smoke-test setup: **37 keypoints, iterative PnP, no RANSAC or keypoint confidence filtering**. No model training takes place here.

In [1]:
!pip -q install ultralytics opencv-python-headless pandas

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

repo = Path('/content/spacecraft-pose-estimation')
if not (repo / 'src' / 'pnp.py').is_file():
    subprocess.run(
        ['git', 'clone', '--quiet', 'https://github.com/EliasNoorzad/spacecraft-pose-estimation.git', str(repo)],
        check=True,
    )
sys.path.insert(0, str(repo / 'src'))
from pnp import solve_pnp

Mounted at /content/drive


In [2]:
import cv2
import json
import numpy as np
import pandas as pd
from IPython.display import display
from ultralytics import YOLO

root = Path('/content/drive/MyDrive/hst_yolo_pose')
test_dir = Path('/content/hst_yolo_pose/images/test')
if not test_dir.is_dir():
    test_dir = root / 'images/test'
model_path = Path('/content/drive/MyDrive/HST_YOLO_Runs/hst_pose_sigma_010/weights/best.pt')

assert test_dir.is_dir(), f'Test images not found: {test_dir}'
assert model_path.is_file(), f'Model not found: {model_path}'

with (root / '_CameraIntrinsicMatrix.json').open() as f:
    K = np.asarray(json.load(f), dtype=np.float64)
with (root / '_HST_Keypoints.json').open() as f:
    points_3d = np.asarray([[p['X'], p['Y'], p['Z']] for p in json.load(f)], dtype=np.float64)
assert K.shape == (3, 3) and points_3d.shape == (37, 3)

with (root / '0000000_metadata.json').open() as f:
    metadata = json.load(f)

def get_frames(obj):
    if isinstance(obj, dict):
        if 'ImagePath' in obj and 'CameraTransform' in obj and 'TargetKeypoints' in obj:
            yield obj
        else:
            for value in obj.values():
                yield from get_frames(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from get_frames(value)

gt_by_name = {Path(frame['ImagePath']).stem: frame for frame in get_frames(metadata)}
print(f'Test images: {len(list(test_dir.glob("*.png")))} | Metadata frames: {len(gt_by_name)}')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Test images: 100 | Metadata frames: 1000


In [3]:
model = YOLO(str(model_path))
predictions = model.predict(
    source=str(test_dir),
    imgsz=640,
    device=0,
    save=False,
    verbose=False,
)

pnp_poses = []
for result in predictions:
    if result.keypoints is None or len(result.keypoints.xy) == 0:
        continue

    points_2d = result.keypoints.xy[0].cpu().numpy()
    if not np.isfinite(points_2d).all():
        continue

    # The geometry solver lives in src/pnp.py. We use all 37 predictions.
    estimate = solve_pnp(points_2d, points_3d, K)
    if estimate is None:
        continue
    R, t = estimate
    pnp_poses.append({'image': Path(result.path).name, 'R': R, 't': t})

print(f'PnP succeeded for {len(pnp_poses)}/{len(predictions)} images')
for pose in pnp_poses[:5]:
    print(f"\n{pose['image']}")
    print('Translation:', pose['t'])
    print('Rotation matrix:\n', pose['R'])

PnP succeeded for 99/100 images

output_0000006.png
Translation: [       2.19     0.07742      28.378]
Rotation matrix:
 [[  -0.067967    -0.89255     -0.4458]
 [   -0.99497    0.093607   -0.035719]
 [   0.073611     0.44113    -0.89442]]

output_0000025.png
Translation: [      1.699     -7.8731      40.054]
Rotation matrix:
 [[    0.63048    -0.40414    -0.66269]
 [    0.58922     0.80497    0.069677]
 [    0.50529     -0.4344     0.74564]]

output_0000027.png
Translation: [    0.80162      1.1267      28.166]
Rotation matrix:
 [[    -0.1513     0.91541      -0.373]
 [   -0.67641    -0.37104    -0.63624]
 [   -0.72082     0.15604     0.67533]]

output_0000030.png
Translation: [    -9.1589     -16.208      53.063]
Rotation matrix:
 [[    0.85368    -0.19698    -0.48211]
 [  -0.046396     -0.9508     0.30631]
 [   -0.51872    -0.23913    -0.82082]]

output_0000032.png
Translation: [    -4.5047       2.284      36.488]
Rotation matrix:
 [[   -0.43656    -0.27508    -0.85659]
 [     0.874

In [4]:
def matrix4(values):
    return np.asarray(
        [[values[f'M{i}{j}'] for j in range(1, 5)] for i in range(1, 5)],
        dtype=np.float64,
    )

def ground_truth_pose(frame):
    # DLVS3-HST stores row-vector transforms and OpenCV uses another camera axis convention.
    camera = matrix4(frame['CameraTransform'])
    target = matrix4(frame['TargetTransform'])
    relative = target @ camera
    flip_axes = np.diag([1.0, -1.0, -1.0])
    return flip_axes @ relative[:3, :3].T, flip_axes @ relative[3, :3]

def rotation_error(R_pred, R_true):
    cosine = (np.trace(R_pred @ R_true.T) - 1) / 2
    return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

# Sanity-check the ground-truth transform against the annotated 2D keypoints.
reprojection_errors = []
for pose in pnp_poses:
    frame = gt_by_name.get(Path(pose['image']).stem)
    if frame is None:
        continue
    R_gt, t_gt = ground_truth_pose(frame)
    gt_2d = np.asarray([[p['X'], p['Y']] for p in frame['TargetKeypoints']], dtype=np.float64)
    rvec_gt, _ = cv2.Rodrigues(R_gt)
    projected, _ = cv2.projectPoints(points_3d, rvec_gt, t_gt, K, None)
    projected = projected.reshape(-1, 2)
    valid = np.isfinite(gt_2d).all(axis=1) & np.isfinite(projected).all(axis=1)
    reprojection_errors.extend(np.linalg.norm(projected[valid] - gt_2d[valid], axis=1))

assert reprojection_errors, 'No matching metadata was found'
median_reprojection = np.median(reprojection_errors)
print(f'Ground-truth reprojection check: {median_reprojection:.4f} px')
if median_reprojection > 2:
    raise ValueError('Ground-truth coordinate convention does not match the annotations')

Ground-truth reprojection check: 0.0009 px


In [5]:
records = []
for pose in pnp_poses:
    frame = gt_by_name.get(Path(pose['image']).stem)
    if frame is None:
        continue
    R_gt, t_gt = ground_truth_pose(frame)
    records.append({
        'Image': pose['image'],
        'Translation error (m)': np.linalg.norm(pose['t'] - t_gt),
        'Rotation error (deg)': rotation_error(pose['R'], R_gt),
    })

pose_comparison = pd.DataFrame(records)
assert not pose_comparison.empty, 'No PnP estimates matched the metadata'

def summarize(df):
    t = df['Translation error (m)']
    r = df['Rotation error (deg)']
    return pd.DataFrame({
        'Metric': ['Mean error', 'Median error', '90th percentile', 'Maximum error'],
        'Translation (m)': [t.mean(), t.median(), t.quantile(.9), t.max()],
        'Rotation (degrees)': [r.mean(), r.median(), r.quantile(.9), r.max()],
    })

print(f'YOLO keypoints + PnP: {len(pose_comparison)} images')
display(summarize(pose_comparison).round(4))

YOLO keypoints + PnP: 99 images


,Metric,Translation (m),Rotation (degrees)
0,Mean error,5.9937,36.5437
1,Median error,1.6720,16.1845
2,90th percentile,15.0623,134.8385
3,Maximum error,77.7202,176.3775


In [6]:
gt_records = []

# Use the same successful image set for a direct comparison.
for pose in pnp_poses:
    frame = gt_by_name.get(Path(pose['image']).stem)
    if frame is None:
        continue
    gt_2d = np.asarray([[p['X'], p['Y']] for p in frame['TargetKeypoints']], dtype=np.float64)
    if not np.isfinite(gt_2d).all():
        continue
    estimate = solve_pnp(gt_2d, points_3d, K)
    if estimate is None:
        continue
    R_pred, t_pred = estimate
    R_gt, t_gt = ground_truth_pose(frame)
    gt_records.append({
        'Image': pose['image'],
        'Translation error (m)': np.linalg.norm(t_pred - t_gt),
        'Rotation error (deg)': rotation_error(R_pred, R_gt),
    })

gt_comparison = pd.DataFrame(gt_records)
assert not gt_comparison.empty, 'No ground-truth PnP estimates were produced'
print(f'Ground-truth keypoints + PnP: {len(gt_comparison)}/{len(pnp_poses)} images')
display(summarize(gt_comparison).round(6))

Ground-truth keypoints + PnP: 99/99 images


,Metric,Translation (m),Rotation (degrees)
0,Mean error,0.000023,0.010814
1,Median error,0.000018,0.007778
2,90th percentile,0.000046,0.025502
3,Maximum error,0.000075,0.040471


### Results from the completed smoke test

These are **previously recorded results**, retained from the original executed Colab notebook. They are not new outputs from this cleaned notebook; running the cells above will recompute them.

| Metric | YOLO keypoints + PnP | Ground-truth keypoints + PnP |
|---|---:|---:|
| Successful images | 99/100 | 99/99 attempted |
| Mean translation error | 5.9937 m | 0.000023 m |
| Median translation error | 1.6720 m | 0.000018 m |
| Mean rotation error | 36.5437° | 0.010814° |
| Median rotation error | 16.1845° | 0.007778° |

Ground-truth reprojection check: **0.0009 px**.